# Handoff Agents

In the previous notebook, 7_multi_agent, I found that it stopped writing to disk.  

Handoff agents receive the contexty from an agent, but can be defined to perform a single task.  My logic is that if an agent has one thing to do, it will probably do it.  "You had one job...One job!"



In [ ]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from openai import AsyncOpenAI
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from instructions_multi_agent import TripPlannerInstructions

# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))

Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']


### The evaluation and wrire_file agent instructions

The evaluation agent remains the same as the previous notebook.

I had to go through a couple if iterations of the instructions to give to the handoff agent that is responsibible for writing the file to disk.  It needs
Very specific instructions to save the correct content to disk.

In [ ]:
sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "output"))
output_file=os.path.join(sandbox_path, "8_trip_plan_using_handoffs.md")
# Generate custom instructions for the trip planner agent

transportation_evaluation_agent_description = "A helpful assistant that evaluates the quality of trip plans based on user preferences and criteria."

transportation_evaluation_agent_instructions = """You are an expert trip planner evaluator. 
Your task is to assess the quality of trip plans, focusing on how clearly the transportation from location to location is described,
There should be clear instructions on how to get from place to place, including modes of transportation, estimated travel times, and any necessary transfers or connections.
This applies for all locations in the trip plan.
If the trip plan lacks clear transportation details, provide constructive feedback on how to improve it.
If the transportation details are clear and sufficient, respond with "The transportation details in the trip plan are clear and sufficient." """


from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

writefile_evaluation_agent_description = "A helpful assistant that records the output that was performed by other agents to disk."

writefile_evaluation_agent_instructions = f"""{RECOMMENDED_PROMPT_PREFIX}
A set of agents have created a trip plan for a client.  The trip plan will be in markdown format and is available in the context that has been given to you.  It is important that you 
find the complete trip plan in the context as this is what needs to be saved to disk.  Once you have determined the full markdown contents of the trip plan, it needs to be saved to 
disk so it can be shared with the client.  Utilize the list_allowed_directories tool to determine where you can save files.  Use the
write_file tool to save the complete itinerary in markdown format to a file: '{output_file}'.  Be sure to only save the appropriate markdown content and not any other text or commentary."""

print(writefile_evaluation_agent_instructions  )

# System context
You are part of a multi-agent system called the Agents SDK, designed to make agent coordination and execution easy. Agents uses two primary abstraction: **Agents** and **Handoffs**. An agent encompasses instructions and tools and can hand off a conversation to another agent when appropriate. Handoffs are achieved by calling a handoff function, generally named `transfer_to_<agent_name>`. Transfers between agents are handled seamlessly in the background; do not mention or draw attention to these transfers in your conversation with the user.

You will be given a trip plan based on our client's preferences.  The trip plan needs to be saved to disk so it can be shared with the client.  Utilze the list_allowed_directories tool to determine where you can save files.  Use the
write_file tool to save the complete itinerary in markdown format to a file: '/media/nathan/linux_ssd/github/agentic_ai_trip_planner/openai_agents_sdk/output/8_trip_plan_using_handoffs.md'.  Be sure to on

## Adding a new agent as a tool

In the code below, the evaluation agent is created and added as a tool to the manager agent.  The manager agent then interacts with the tools, based on the instructions given.  Here, the manager agent will use the evaluation agent to validate the the transportion information in the trip plan is acceptable before finishing.

This one is hit or miss.  I found that if I use the local gpt-oss model on ollama, it can lose track of what it is doing, making the agent behave inconsistently.  If I connect to Open AI, I hit throttling limits (tokens per minute).  So, I will have to refine the agent architecture in the next notebook...

In [3]:
planner = TripPlannerInstructions(
    output_file=output_file
)

custom_instructions = planner.get_instructions()
# Update instructions to force the handoff
custom_instructions = custom_instructions.replace("using the write_file tool", "by handing off to the write_file_agent")

print(custom_instructions)
print("*" * 120)

# Set the base_url to your local Ollama instance
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Set a dummy API key (required by the SDK, but not used by Ollama)
DUMMY_API_KEY = "ollama"

# Initialize the AsyncOpenAI client with the custom base_url
client = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=DUMMY_API_KEY,
)

# Specify the model you pulled with Ollama
#OLLAMA_MODEL_NAME = "gpt-oss:20b" 
OLLAMA_MODEL_NAME = "gpt-oss_131k_context:20b" 

# Wrap the client in the Agents SDK model class
model = OpenAIChatCompletionsModel(
    openai_client=client,
    model=OLLAMA_MODEL_NAME
)

filesystem_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}
serper_params = {"command": "uvx", "args": ["serper-mcp-server"], "env": {"SERPER_API_KEY": os.environ.get('SERPER_API_KEY')}}

#web_search_tool = WebSearchTool(search_context_size="low")

async with MCPServerStdio(params=filesystem_params, client_session_timeout_seconds=30) as mcp_server_files:
    async with MCPServerStdio(params=serper_params, client_session_timeout_seconds=45) as mcp_server_serper:

        # Define the transportation evaluation expert agent
        transportation_evaluation_agent = Agent(
            model=model,
            name="transportation evaluation expert agent",
            instructions=transportation_evaluation_agent_instructions
        )

        # Define the write file agent
        writefile_agent = Agent(
            model=model,
            name="write_file_agent",
            instructions=writefile_evaluation_agent_instructions,
            mcp_servers=[mcp_server_files]
        )

        # Define the trip planner agent with the transportation evaluation expert as a tool
        trip_planner_agent = Agent(
            model=model,
            name="Trip Planner Agent",
            instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities.",
            mcp_servers=[mcp_server_serper],
            tools=[
                transportation_evaluation_agent.as_tool(
                    tool_name="transportation_evaluation_expert",
                    tool_description=transportation_evaluation_agent_description,    
                )
            ],
            handoffs=[writefile_agent]
        )
        with trace("Trip Planner Agent Ollama"):
            result = await Runner.run(trip_planner_agent, custom_instructions, max_turns=50)
            print(result.final_output)

You are a methodical and detail-oriented trip planning assistant. Your task is to create a COMPLETE, timeline-based trip itinerary with specific departure/arrival times and durations for every activity.

CRITICAL: You must complete the ENTIRE itinerary before finishing. Do not stop at research phase. Do not ask for permission to continue. Work through all steps until you have a fully detailed day-by-day schedule.

The customer has provided the following details for their trip:
- Home Location: Columbus, OH
- Departure Date: 2026/02/25
- Return Date: 2026/03/07
- Destination: Tokyo, Japan
- Must-Do Activities: Visit the Tokyo Tower, Explore Akihabara, Experience a traditional tea ceremony, Visit the Tsukiji Fish Market, Take a day trip to Mount Fuji   
- Number of Travelers: 3
- Ages of Travelers: 51, 50, 17
- Other Considerations: 
  - I will be running in the Tokyo marathon on Sunday March 1, so I only need a relaxing place to eat on that day.
  - Starting on March 3, throughout the r